# 环节 10 · 推理解码与 KV Cache（配套 Notebook）

> 配套长文：[环节10-推理解码与KV缓存详解.md](./环节10-推理解码与KV缓存详解.md)
> 定位：把"Prefill 与 Decode 瓶颈不同""KV Cache 省了多少""采样参数怎么改输出""为什么必须有 max_tokens"全部算成数。纯 Python 标准库，零依赖。

| 本 Notebook | 长文章节 | 验证什么 |
|---|---|---|
| §1 两阶段的性格差异 | §1 | Prefill 吃算力、Decode 吃带宽 |
| §2 KV Cache 的账 | §2 | 只增不改；省 667 倍；尺寸与容量判据 |
| §3 采样方法 | §3 | greedy / temperature / top-k / top-p |
| §4 停止条件 | §4 | EOS 不可靠，max_tokens 是唯一保证 |


## 1. Prefill 与 Decode：瓶颈完全不同（长文 §1）

| | 干什么 | 瓶颈 |
|---|---|---|
| **Prefill** | 整段 prompt 并行前向，填满 KV Cache，出第 1 个 token | 计算密集（大矩阵乘） |
| **Decode** | 逐 token：只算新位置，与 KV Cache 做注意力 | **访存密集**（每步把权重整体搬一遍） |

用一笔"搬运量"的账看清楚为什么 decode 慢：


In [ ]:
d_model, n_layers = 4096, 32
w_bytes = 2                        # FP16


def weight_bytes():
    """一遍前向要把多少权重字节搬过内存：每层 ≈ 16d²（环节10 §2.4 心算口径）。"""
    return n_layers * 16 * d_model ** 2 * w_bytes


prompt_len, generated = 512, 512
wb = weight_bytes()
print(f"模型权重 ≈ {wb/1e9:.1f} GB（每层 16d² × {n_layers} 层 × {w_bytes}B）\n")

print("Prefill（一次算完整个 prompt）：")
print(f"  算力：{prompt_len} 个位置一起参与矩阵乘 → GPU 算力吃满")
print(f"  在这一步里，权重只搬了 1 遍（{wb/1e9:.1f} GB）")
print(f"  平均每个 token 摊到的权重搬运 = {wb/prompt_len/1e6:.1f} MB/token")

print(f"\nDecode（逐 token 生成 {generated} 个）：")
print(f"  算力：每步只有 1 行 query → GPU 算力基本闲置")
print(f"  但每步都要把权重整体搬一遍：{generated} 步 × {wb/1e9:.1f} GB"
      f" = {generated*wb/1e9:,.0f} GB")
print(f"  平均每个 token 摊到的权重搬运 = {wb/1e6:.1f} MB/token")

print(f"\n→ 同样生成 1 个 token，Decode 的权重搬运量是 Prefill 的"
      f" {wb/(wb/prompt_len):.0f} 倍 —— 这就是“访存密集”。")
print("  所以提高吞吐的核心不是“把单请求算更快”，而是**把更多请求塞进同一批**")
print("  （batch 里 N 个请求合读同一份权重 = 读一次喂 N 人，见环节 11 §5.2）。")


## 2. KV Cache：只增不改的日志（长文 §2.2）

**为什么只缓存 K/V、从不缓存 Q**：位置 i 的 Q 只用一次（算自己那一行打分），使命即终；而 K_i / V_i 会被之后每一个新 Q 反复读。


In [ ]:
tokens = ["a", "b", "c"]
k_cache, v_cache = [], []
print("模拟：输入 [a] 起步，逐 token 生成（每步只算新 token 的 K/V）\n")
print(f"{'步':>3} {'输入':>5} {'新算并缓存':>12} {'缓存长度':>9}   这一行 Q 要与谁做注意力")
print("-" * 68)
for step, tok in enumerate(tokens, start=1):
    k_cache.append(f"k_{tok}")
    v_cache.append(f"v_{tok}")
    hist = "无历史（或仅自己）" if len(k_cache) == 1 else f"缓存 {k_cache}"
    print(f"{step:>3} {tok:>5} {f'k_{tok}, v_{tok}':>12} {len(k_cache):>9}   {hist}")

print(f"\n缓存形状演化：({len(k_cache)}, d_k) —— 每次 Decode **追加一行**，旧行永不改写")
print("  S = q_new · K_cacheᵀ / √d_k   只算 1×T（长文 §2.2 的图）")
print("\n为什么只增不改（长文 §2.2 的两个保证）：")
print("  ① 推理时 W_K/W_V 冻结 → 投影函数恒定；")
print("  ② 因果 Mask 下位置 i 只依赖 prefix ≤ i，之后没有任何计算路径能回头改写它。")
print("  反例：训练时参数每步都在更新，旧 K/V 全部作废 → 所以**训练从不缓存**。")


In [ ]:
print("有/无 KV Cache 的代价对比（生成 n 个 token 的注意力点积次数）：\n")
print(f"{'n':>6} {'不缓存（每步重算全前缀）':>24} {'用 KV Cache':>16} {'省':>10}")
print("-" * 62)
for n in (100, 500, 1000, 2000):
    no_cache = sum(t * t for t in range(1, n + 1))        # 第 t 步重算 t×t
    cache = sum(t for t in range(1, n + 1))               # 第 t 步 1×t
    print(f"{n:>6} {no_cache:>24,} {cache:>16,} {no_cache/cache:>9.0f}x")

print("\n→ 不缓存：每步 O(T²)、总共 O(n³)；缓存后：每步 O(T)、总共 O(n²)。")
print("  代价是显存：KV Cache ≈ 2 × 层数 × KV头数 × 头维度 × 总长度 × 字节数 —— 用显存换算力。")


## 3. KV 尺寸与容量判据（长文 §2.4）

心算核心：把总公式拆成 **单 token 常数 × 上下文长度**。


In [ ]:
print(f"{'模型':<20} {'结构':<14} {'单 token KV':>14} {'8K 上下文':>12}")
print("-" * 66)
llama2_7b = 2 * 32 * 32 * 128 * 2          # 32 层、32 个 KV 头（MHA）、head_dim 128、FP16
for name, tag, layers, kvh, hd in [
    ("Llama-2 7B", "MHA(32 KV头)", 32, 32, 128),
    ("Llama-3 8B", "GQA(8 KV头)", 32, 8, 128),
    ("Llama-3 70B", "GQA(8 KV头)", 80, 8, 128),
]:
    per_tok = 2 * layers * kvh * hd * 2
    print(f"{name:<20} {tag:<14} {per_tok/1024:>11.0f} KB {per_tok*8192/1e9:>10.2f} GB")

print("\n容量判据（长文 §2.4 最有工程价值的一句）：")
kv_128k = llama2_7b * 131072
print(f"  MHA 7B 跑到 128K 上下文 → KV = {kv_128k/1e9:.1f} GB")
print(f"  权重只有 14 GB → **KV 是权重的 {kv_128k/1e9/14:.1f} 倍**（长文 4.6 倍）")
print("  → 长上下文里，KV Cache 才是显存第一大头，不是权重。")

print("\n反算并发上限：并发 ≤ (可用显存 − 权重) / (平均上下文 × 单 token KV)")
for ctx in (1024, 4096, 8192):
    avail = 7e9                                    # 假设可用 KV 7 GB
    print(f"  平均上下文 {ctx:>5} → 可并发 {avail/(ctx*llama2_7b):>6.1f} 个请求")

print("\nGQA 的价值：把单 token KV 从 512 KB 降到 128 KB（4 倍），")
print("  反超拐点也相应推迟 4 倍（长文 §2.4）。")


## 4. 采样：同一份 logits，四种味儿（长文 §3）

计算链：`logits → temperature 缩放 → top-k / top-p 截断 → softmax → 按概率抽 1 个`。


In [ ]:
import math
import random

logits = [3.2, 2.8, 1.5, 0.9, 0.3, -1.0, -2.5, -4.0]
VOCAB = list("abcdefgh")


def softmax_scaled(v, T=1.0, keep=None):
    v = [x / T for x in v]
    if keep is not None:
        v = [x if i in keep else float("-inf") for i, x in enumerate(v)]
    m = max(x for x in v if x != float("-inf"))
    e = [0.0 if x == float("-inf") else math.exp(x - m) for x in v]
    s = sum(e)
    return [x / s for x in e]


def entropy(p):
    return -sum(x * math.log(x) for x in p if x > 0)


p0 = softmax_scaled(logits)
print("① Greedy（argmax）：", VOCAB[max(range(len(logits)), key=lambda i: logits[i])],
      " —— 确定性，永远取第一个")
print(f"\n② Temperature（只改分布形状，候选集不变）：")
print(f"{'T':>6} {'top1':>10} {'熵':>10}   说明")
for T in (0.3, 0.5, 1.0, 2.0):
    p = softmax_scaled(logits, T)
    note = {0.3: "很陡，接近 argmax", 0.5: "偏保守", 1.0: "原始分布", 2.0: "拉平，更有创意"}[T]
    print(f"{T:>6} {max(p):>10.4f} {entropy(p):>10.4f}   {note}")
print(f"  原始熵 {entropy(p0):.4f}；极端情况：T→0 退化成 argmax，T→∞ 退化成均匀分布")

srt = sorted(range(len(logits)), key=lambda i: -logits[i])
print(f"\n③ Top-k（只在概率前 k 个里归一化）：")
for k in (2, 4, 6):
    keep = set(srt[:k])
    p = softmax_scaled(logits, keep=keep)
    print(f"  k={k}: 候选 {[VOCAB[i] for i in srt[:k]]}"
          f"  → 概率 {[round(p[i], 4) for i in srt[:k]]}")

print(f"\n④ Top-p / nucleus（候选数随分布自适应）：")
for pp in (0.9, 0.95):
    keep, acc = [], 0.0
    for i in srt:
        keep.append(i)
        acc += p0[i]
        if acc >= pp:
            break
    p = softmax_scaled(logits, keep=set(keep))
    print(f"  p={pp}: 需要 {len(keep)} 个候选 {[VOCAB[i] for i in keep]}"
          f"  → {[round(p[i], 4) for i in keep]}")

print("\n工程口径（长文 §3）：")
print("  结构化/代码 → T=0 或约束解码；创意 → T≈0.9~1.2 + top-p（与 temperature **二选一**）；")
print("  要完全复现必须固定 seed。")


In [ ]:
# 采样是概率性的：同一个分布，多次采样结果不同（seed 固定才可复现）


def sample_once(p, rng):
    r, acc = rng.random(), 0.0
    for i, x in enumerate(p):
        acc += x
        if r <= acc:
            return VOCAB[i]
    return VOCAB[-1]


for seed in (0, 1):
    rng = random.Random(seed)
    draws = [sample_once(p0, rng) for _ in range(20)]
    print(f"seed={seed}: 20 次采样结果 = {''.join(draws)}")

print("\n→ 分布一样，但采样带随机性；要“完全复现”必须固定 seed")
print("  （长文 §3 工程落点：确定性任务用 greedy 或 T=0）。")


## 5. 什么时候停：三个信号（长文 §4.1）

| # | 信号 | 谁在判 | 可靠吗 |
|---|---|---|---|
| ① | EOS token | 模型（训练学来） | **不保证**——可能永不吐 EOS |
| ② | max_tokens | 服务端 | **唯一强制保证** |
| ③ | stop 序列 / 客户端中断 | 应用层 | 由你控制 |

下面把 decode 循环写出来，看 `finish_reason` 是怎么产生的：


In [ ]:
EOS = "<eos>"


def decode_loop(dist_fn, max_tokens, stop_words=()):
    """模拟一次请求的 decode 循环：每步采样 → 三连检查 → 决定是否退出。"""
    out, reason = [], None
    for step in range(1, max_tokens + 1):
        nxt = dist_fn(step)                      # 假装的模型输出
        if nxt == EOS:
            reason = "stop"                      # ① 模型自己说完了
            break
        if nxt in stop_words:
            reason = "stop"                      # ③ 命中调用方指定的停止词
            break
        out.append(nxt)
    if reason is None:
        reason = "length"                        # ② 撞到 max_tokens 上限
    return out, reason


def normal_model(step):
    return EOS if step == 5 else f"t{step}"


def chatty_model(step):
    return f"t{step}"                            # 永远不吐 EOS（“话痨”模型）


for name, fn, mt in [("正常模型 · max_tokens=50", normal_model, 50),
                     ("话痨模型 · max_tokens=50", chatty_model, 50),
                     ("话痨模型 · max_tokens=8", chatty_model, 8)]:
    out, reason = decode_loop(fn, mt)
    print(f"{name:<28} 生成 {len(out):>2} 个 token → finish_reason = '{reason}'")
    print(f"{'':<28} 输出 {out}")

print("\n→ EOS 靠模型自愿（不可靠）；max_tokens 是工程兜底（唯一保证）。")
print("  拿到 finish_reason='length' 说明被截断没说完 —— 续写 / 加大上限 / 缩短 prompt（长文 §4.1）。")

print("\nmax_tokens 语义提醒（长文 §4.2）：")
for ctx, prompt_len, mt in [(8192, 3000, 4096), (8192, 7000, 4096)]:
    total = prompt_len + mt
    ok = "✅" if total <= ctx else "❌ 超上下文 → 必然被截断"
    print(f"  context={ctx}, prompt={prompt_len}, max_tokens={mt}"
          f" → 总消耗 {total} {ok}")
print("  → max_tokens 只数**生成**的 token，不含 prompt；合理上限 = context − prompt − 余量")
print("  已知默认坑：vLLM 默认 16、HF generate 默认 max_length=20（含 prompt）、Ollama 默认 128")


## 6. 自测（长文 §6）

| 问题 | 本 Notebook 的现场证据 |
|---|---|
| Prefill / Decode 瓶颈分别是什么？ | §1：Decode 每 token 要把权重整体搬一遍 |
| 为什么只缓存 K/V 不缓存 Q？ | §2：Q 用完即弃，K/V 被后续每个新 Q 反复读 |
| KV Cache 省了多少？ | §2：n=1000 时点积次数省 667 倍 |
| KV 大小怎么算？ | §3：`2×层×KV头×头维×长度×字节` |
| 长上下文里谁是大头？ | §3：MHA 7B @128K → KV 是权重的 4.6 倍 |
| temperature 与 top-p 能一起调吗？ | §4：官方口径二选一，避免双重干预分布 |
| 为什么必须有 max_tokens？ | §5：EOS 不保证，只有它是强制兜底 |

**上一站** [环节 09 · 训练管线](./环节09-训练管线详解.md)   **下一站** [环节 11 · 服务化与推理引擎](./环节11-服务化与推理引擎详解.md)
